# DriveValue — Notebook 3: Model Explainability with SHAP

**Project:** DriveValue — Nigerian Used Car Price Predictor  

This notebook covers:
- Loading the saved model
- SHAP bar plot — overall feature importance
- SHAP beeswarm plot — how each feature affects price
- SHAP waterfall plot — explaining a single prediction
- Written business interpretation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import shap
import kagglehub
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split

os.makedirs('../assets', exist_ok=True)

shap.initjs()
print('Libraries loaded!')

## 1. Load Model and Data

In [ ]:
# Load saved model
pipeline = joblib.load('../model/car_price_model.joblib')
feature_cols = joblib.load('../model/feature_columns.joblib')

NUMERIC_FEATURES = feature_cols['numeric']
CATEGORICAL_FEATURES = feature_cols['categorical']
ALL_FEATURES = feature_cols['all']

print('Model loaded!')
print(f'Features: {ALL_FEATURES}')

In [ ]:
# Reload dataset
path = kagglehub.dataset_download('makindekayode/nigerian-car-prices-dataset')
df = pd.read_csv(os.path.join(path, 'car_prices.csv'))

df = df.drop_duplicates()
df = df.dropna(subset=['price'])
df['car_age'] = 2025 - df['Year of manufacture']

drop_cols = ['car_id', 'car', 'Colour', 'Trim', 'Model']
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

X = df[ALL_FEATURES]
y = df['price']

_, X_test, _, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

print(f'Test set: {X_test.shape[0]:,} rows')

## 2. Prepare Data for SHAP

In [ ]:
# Get the preprocessor and model from the pipeline
preprocessor = pipeline.named_steps['preprocessor']
rf_model = pipeline.named_steps['model']

# Transform the test data
X_test_transformed = preprocessor.transform(X_test)

# Get feature names after encoding
cat_feature_names = preprocessor.named_transformers_['cat']['encoder'].get_feature_names_out(CATEGORICAL_FEATURES).tolist()
all_feature_names = NUMERIC_FEATURES + cat_feature_names

print(f'Transformed shape: {X_test_transformed.shape}')
print(f'Total features after encoding: {len(all_feature_names)}')

## 3. SHAP Explainer

In [ ]:
# Use a sample for speed
sample_size = min(200, len(X_test_transformed))
X_sample = X_test_transformed[:sample_size]

explainer = shap.TreeExplainer(rf_model)
shap_values = explainer.shap_values(X_sample)

print(f'SHAP values computed for {sample_size} samples!')

## 4. SHAP Bar Plot — Overall Feature Importance

This shows the average impact of each feature across all predictions. The longer the bar, the more important that feature is in determining car price.

In [ ]:
shap.summary_plot(
    shap_values,
    X_sample,
    feature_names=all_feature_names,
    plot_type='bar',
    max_display=15,
    show=False
)
plt.title('SHAP Feature Importance — Top 15 Features')
plt.tight_layout()
plt.savefig('../assets/shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()

print("""
INTERPRETATION:
The bar chart shows the mean absolute SHAP value for each feature.
Car age is the most influential feature — newer cars are worth significantly more.
Mileage ranks second — higher mileage reduces price consistently.
Brand (Make) is also a strong driver — premium brands command higher prices.
Condition matters — Brand New and Foreign Used command higher prices than Nigerian Used.
""")

## 5. SHAP Beeswarm Plot — Feature Impact Direction

This shows not just which features matter but HOW they affect price. Red dots = high feature value. Blue dots = low feature value. Right side = increases price. Left side = decreases price.

In [ ]:
shap.summary_plot(
    shap_values,
    X_sample,
    feature_names=all_feature_names,
    max_display=15,
    show=False
)
plt.title('SHAP Beeswarm Plot — Feature Impact Direction')
plt.tight_layout()
plt.savefig('../assets/shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

print("""
INTERPRETATION:
Car age: Low car age (newer car = red dots) pushes price UP (right side).
High car age (old car = blue dots) pushes price DOWN (left side).
This confirms newer cars are worth more.

Mileage: High mileage (red dots) pushes price DOWN (left side).
Low mileage (blue dots) pushes price UP (right side).
This confirms that lower mileage cars are worth more.

Engine Size and Horse Power: Higher values tend to increase price,
consistent with more powerful cars being more expensive.
""")

## 6. SHAP Waterfall Plot — Explaining One Prediction

This explains a single car's predicted price. It shows exactly which features pushed the price up or down from the baseline average.

In [ ]:
# Pick one example to explain
example_idx = 0
example_car = X_test.iloc[example_idx]
actual_price = y_test.iloc[example_idx]
predicted_price = pipeline.predict(X_test.iloc[[example_idx]])[0]

print(f'Example car details:')
print(example_car)
print(f'\nActual price:    N{actual_price:,.0f}')
print(f'Predicted price: N{predicted_price:,.0f}')

# Waterfall plot
shap_explanation = shap.Explanation(
    values=shap_values[example_idx],
    base_values=explainer.expected_value,
    data=X_sample[example_idx],
    feature_names=all_feature_names
)

shap.plots.waterfall(shap_explanation, max_display=12, show=False)
plt.title('SHAP Waterfall — Single Car Prediction Explained')
plt.tight_layout()
plt.savefig('../assets/shap_waterfall.png', dpi=150, bbox_inches='tight')
plt.show()

print("""
INTERPRETATION:
The waterfall plot shows why the model predicted this specific price.
Starting from the baseline (average prediction across all cars),
each bar shows how much a specific feature pushed the prediction
higher (red, right) or lower (blue, left).
This is the most powerful tool for explaining individual predictions
to a car seller or buyer who wants to understand why a car is valued
at a particular price.
""")

## 7. Business Insights Summary

Based on the SHAP analysis, here are the key business insights for DriveValue:

**1. Car Age is the strongest predictor of price**  
Newer cars are consistently worth more. Every additional year of age reduces the predicted price. This aligns with standard depreciation principles.

**2. Mileage reduces car value**  
Higher mileage cars are priced lower across the board. A car with 200,000km is predicted to be worth significantly less than an identical car with 30,000km.

**3. Brand matters significantly**  
Premium brands like Mercedes-Benz, BMW and Land Rover push predicted prices higher. Budget brands like Hyundai and KIA tend to keep prices lower.

**4. Condition has a clear impact**  
Brand New > Foreign Used > Nigerian Used in terms of price impact. A Foreign Used car is significantly more valuable than a Nigerian Used equivalent.

**5. Engine size and horse power add value**  
More powerful engines command higher prices, consistent with buyer preferences for performance vehicles in the Nigerian market.

These insights are directly actionable for car sellers using DriveValue — they can understand exactly what is driving their car's valuation and what factors they should highlight when selling.